# 04 — Imports, Introspection, and Plugin Systems

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
# Provided for this lesson.
from collections.abc import Callable, Iterable

from lesson import Exercises, ___, run

check = Exercises('advanced/04', {1: '7e505b96e1b4ed86', 2: '89d81a04e23dd21a', 3: 'd8ef3d8bdffe93de', 4: 'f8c2879ecb7e948b', 5: 'd81bf19dcb7053c3', 6: '58a70cf9b55e71ed', 7: '578544717906c411', 8: '3266231244b5b9cb'}, practice='advanced_01_internals_typing_architecture', independent='parameter_defaults')

## 1. Introspection

- `inspect.signature(f).parameters` maps each name to a `Parameter` with `.default` and `.kind`.
- A parameter without a default has `.default is inspect.Parameter.empty`.
- `dir(obj)`, `getattr(obj, name, default)` and `callable(x)` explore objects at runtime.

In [ ]:
import inspect

def connect(host, port=5432, *, timeout=5.0):
    pass

for name, p in inspect.signature(connect).parameters.items():
    print(name, None if p.default is inspect.Parameter.empty else p.default, p.kind.name)

**Exercise 1.** List the parameter names of `connect`.

In [ ]:
import inspect
def connect(host, port=5432, timeout=5.0): pass
names = list(___)   # the parameters of inspect.signature(connect)
check(1, names)

**Exercise 2.** Map each parameter to its default, using `None` when there isn't one.

In [ ]:
import inspect
def f(a, b=2, c="x"): pass
defaults = {name: (None if p.default is inspect.Parameter.empty else ___) for name, p in inspect.signature(f).parameters.items()}
check(2, defaults)

**Exercise 3.** Read an attribute that may be missing, defaulting to `"n/a"`.

In [ ]:
class C: pass
value = getattr(C(), "version", ___)
check(3, value)

## 2. Trusted registries, not eval

- Never run code from names in input (`eval`, `exec`, `importlib` on user text).
- Map approved names to functions in a dict, and check an allowlist before registering.
- Unknown name at run time → `KeyError`; disallowed name at registration → `ValueError`.

In [ ]:
allowed = {"upper", "reverse"}
plugins = {}

def register(name, func):
    if name not in allowed:
        raise ValueError(f"plugin {name!r} is not allowed")
    plugins[name] = func

register("upper", str.upper)
print(plugins["upper"]("hi"))
try:
    register("shell", print)
except ValueError as error:
    print(error)

**Exercise 4.** Only register names in the allowlist.

In [ ]:
allowed = {"upper"}
plugins = {}
for name, func in [("upper", str.upper), ("evil", print)]:
    if name in ___:
        plugins[name] = func
check(4, sorted(plugins))

**Exercise 5.** Run a plugin by name.

In [ ]:
plugins = {"double": lambda x: x * 2, "neg": lambda x: -x}
result = plugins[___](21)
check(5, result)

**Exercise 6.** Running an unregistered name should raise which error? Write its class name.

In [ ]:
answer = ___
check(6, answer)

## 3. Modules and imports

- A module runs once on first import; later imports reuse the cached object in `sys.modules`.
- Keep import-time code free of side effects: define things, don't do work.
- `if __name__ == "__main__":` runs only when the file is executed directly.

In [ ]:
import sys, json
print("json" in sys.modules, sys.modules["json"] is json)
print(__name__)

**Exercise 7.** Is a module imported twice run twice? Write `True` or `False`.

In [ ]:
answer = ___
check(7, answer)

**Exercise 8.** What is `__name__` in a file run directly with `python file.py`?

In [ ]:
answer = ___
check(8, answer)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `PluginRegistry`

Register plugins from an allowlist and run them by name.

In [ ]:
class PluginRegistry:
    """Run plugins only by approved name.

    register(name, func) raises ValueError for names not in the allowlist;
    run(name, *args) raises KeyError for unregistered names.
    """

    def __init__(self, allowed: Iterable[str]):
        self._allowed = ___
        self._plugins: dict[str, Callable] = {}

    def register(self, name: str, func: Callable) -> None:
        if ___:
            raise ValueError(f"plugin {name!r} is not allowed")
        self._plugins[name] = func

    def run(self, name: str, *args):
        if ___:
            raise KeyError(name)
        return self._plugins[name](*args)

    def names(self) -> list[str]:
        return ___

In [ ]:
check('PluginRegistry')

### Independent build — `parameter_defaults`

Return a mapping of every parameter name to its declared default, using None when no default is declared. Include keyword-only parameters.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def parameter_defaults(func: Callable) -> dict[str, object]:
    raise NotImplementedError

In [ ]:
check('parameter_defaults')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** A second `import json` in the same process…

   - **a)** Fails
   - **b)** Reuses the cached module from sys.modules
   - **c)** Re-executes the module

**2.** inspect.signature(f) gives you…

   - **a)** The parameters, defaults and annotations
   - **b)** The call count
   - **c)** The source code

**3.** A safe plugin system runs…

   - **a)** Any module name a user types
   - **b)** Only names on an explicit allowlist
   - **c)** Code from the internet

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/04', {1: 'af6c976d520e', 2: '1a820b8301af', 3: 'be6b892975d0'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [05 — asyncio Event Loops, Coroutines, and Awaitables](05_asyncio_event_loop.ipynb)